In [1]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

In [2]:
# --- Globale Konfiguration & State ---
x_min, x_max = -5, 5
y_min, y_max = -5, 5

points = np.empty((0, 2))
is_updating = False  # Verhindert zirkuläre Callbacks, wenn Slider per Code aktualisiert werden

In [3]:
# --- UI Elemente ---
style = {'description_width': '110px'}

slider_a = widgets.FloatSlider(value=0.0, min=-4.9, max=4.9, step=0.1, description="Grenze a (x):", style=style)
slider_b = widgets.FloatSlider(value=1.0, min=-4.9, max=4.9, step=0.1, description="Grenze b (y):", style=style)

slider_n1 = widgets.IntSlider(value=25, min=0, max=500, description="n(A ∩ B):", style=style)
slider_n2 = widgets.IntSlider(value=25, min=0, max=500, description="n(A - B):", style=style)
slider_n3 = widgets.IntSlider(value=25, min=0, max=500, description="n(B - A):", style=style)
slider_n4 = widgets.IntSlider(value=25, min=0, max=500, description="n(Ω - (A∪B)):", style=style)
slider_omega = widgets.IntSlider(value=100, min=0, max=2000, description="n(Ω) total:", style=style)

def create_read_slider(desc):
    return widgets.FloatSlider(value=0, min=0, max=1, step=0.01, description=desc, disabled=True, style=style)

s_p_A_and_B = create_read_slider("P(A ∩ B)")
s_p_A_x_B   = create_read_slider("P(A) * P(B)")
s_p_A       = create_read_slider("P(A)")
s_p_A_giv_B = create_read_slider("P(A | B)")
s_p_B       = create_read_slider("P(B)")
s_p_B_giv_A = create_read_slider("P(B | A)")
s_p_A_or_B  = create_read_slider("P(A ∪ B)")
s_p_neither = create_read_slider("P(Ω - (A∪B))")

plot_output = widgets.Output()

# --- Logik & Masken ---
def mask_n1(a, b): return (points[:, 0] >= a) & (points[:, 1] <= b)
def mask_n2(a, b): return (points[:, 0] >= a) & (points[:, 1] > b)
def mask_n3(a, b): return (points[:, 0] < a) & (points[:, 1] <= b)
def mask_n4(a, b): return (points[:, 0] < a) & (points[:, 1] > b)

def sync_all_and_plot():
    global is_updating
    is_updating = True
    try:
        a, b = slider_a.value, slider_b.value
        m1, m2, m3, m4 = mask_n1(a, b), mask_n2(a, b), mask_n3(a, b), mask_n4(a, b)
        
        c1, c2, c3, c4 = np.sum(m1), np.sum(m2), np.sum(m3), np.sum(m4)
        total = len(points)
        
        # 1. Slider mit echten Counts synchronisieren
        slider_n1.value = c1
        slider_n2.value = c2
        slider_n3.value = c3
        slider_n4.value = c4
        slider_omega.value = total
        
        # 2. Wahrscheinlichkeiten berechnen
        if total > 0:
            p_A, p_B = (c1 + c2) / total, (c1 + c3) / total
            s_p_A_and_B.value = c1 / total
            s_p_A_x_B.value   = p_A * p_B
            s_p_A.value       = p_A
            s_p_B.value       = p_B
            s_p_A_giv_B.value = c1 / (c1 + c3) if (c1 + c3) > 0 else 0.0
            s_p_B_giv_A.value = c1 / (c1 + c2) if (c1 + c2) > 0 else 0.0
            s_p_A_or_B.value  = (c1 + c2 + c3) / total
            s_p_neither.value = c4 / total
        else:
            for s in [s_p_A_and_B, s_p_A_x_B, s_p_A, s_p_B, s_p_A_giv_B, s_p_B_giv_A, s_p_A_or_B, s_p_neither]:
                s.value = 0.0
                
        # 3. Plot aktualisieren
        with plot_output:
            plot_output.clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(7, 6))
            ax.set_xlim(x_min, x_max)
            ax.set_ylim(y_min, y_max)
            
            ax.axvspan(a, x_max, color='blue', alpha=0.15, label=f'A (x ≥ {a:.1f})')
            ax.axhspan(y_min, b, color='green', alpha=0.15, label=f'B (y ≤ {b:.1f})')
            ax.axvline(x=a, color='black', linestyle='--', alpha=0.5)
            ax.axhline(y=b, color='black', linestyle='--', alpha=0.5)
            
            if total > 0:
                if c1 > 0: ax.scatter(points[m1, 0], points[m1, 1], color='darkcyan', marker='o', edgecolors='k')
                if c2 > 0: ax.scatter(points[m2, 0], points[m2, 1], color='blue', marker='o', edgecolors='k')
                if c3 > 0: ax.scatter(points[m3, 0], points[m3, 1], color='green', marker='o', edgecolors='k')
                if c4 > 0: ax.scatter(points[m4, 0], points[m4, 1], color='gray', marker='x')
                
            ax.set_xlabel("x")
            ax.set_ylabel("y")
            ax.legend(loc='upper left', bbox_to_anchor=(1.05, 1))
            plt.tight_layout()
            plt.show()
    finally:
        is_updating = False

# --- Event Handler ---
def make_region_handler(mask_func, get_xr, get_yr):
    def handler(change):
        global points
        if is_updating: return
        diff = change['new'] - change['old']
        a, b = slider_a.value, slider_b.value
        
        if diff > 0:
            xr, yr = get_xr(a), get_yr(b)
            new_pts = np.column_stack((
                np.random.uniform(xr[0], xr[1], diff),
                np.random.uniform(yr[0], yr[1], diff)
            ))
            points = np.vstack([points, new_pts])
        elif diff < 0:
            indices = np.where(mask_func(a, b))[0]
            if len(indices) > 0:
                to_delete = indices[-min(abs(diff), len(indices)):]
                points = np.delete(points, to_delete, axis=0)
        sync_all_and_plot()
    return handler

def handle_omega(change):
    global points
    if is_updating: return
    diff = change['new'] - change['old']
    if diff > 0:
        new_pts = np.column_stack((
            np.random.uniform(x_min, x_max, diff),
            np.random.uniform(y_min, y_max, diff)
        ))
        points = np.vstack([points, new_pts])
    elif diff < 0:
        del_count = min(abs(diff), len(points))
        if del_count > 0: points = points[:-del_count]
    sync_all_and_plot()

# Initiales Befüllen des States passend zu den Default-Slidern (4x25 = 100)
a_def, b_def = slider_a.value, slider_b.value
points = np.vstack([
    np.column_stack((np.random.uniform(a_def, x_max, slider_n1.value), np.random.uniform(y_min, b_def, slider_n1.value))),
    np.column_stack((np.random.uniform(a_def, x_max, slider_n2.value), np.random.uniform(b_def, y_max, slider_n2.value))),
    np.column_stack((np.random.uniform(x_min, a_def, slider_n3.value), np.random.uniform(y_min, b_def, slider_n3.value))),
    np.column_stack((np.random.uniform(x_min, a_def, slider_n4.value), np.random.uniform(b_def, y_max, slider_n4.value)))
])

# Callbacks registrieren
slider_a.observe(lambda _: sync_all_and_plot(), names='value')
slider_b.observe(lambda _: sync_all_and_plot(), names='value')

slider_n1.observe(make_region_handler(mask_n1, lambda a: (a, x_max), lambda b: (y_min, b)), names='value')
slider_n2.observe(make_region_handler(mask_n2, lambda a: (a, x_max), lambda b: (b, y_max)), names='value')
slider_n3.observe(make_region_handler(mask_n3, lambda a: (x_min, a), lambda b: (y_min, b)), names='value')
slider_n4.observe(make_region_handler(mask_n4, lambda a: (x_min, a), lambda b: (b, y_max)), names='value')
slider_omega.observe(handle_omega, names='value')

# --- Layout (Exakt ausgerichtetes Grid) ---
row1 = widgets.HBox([slider_a, slider_b])
row2 = widgets.HBox([slider_n1, s_p_A_and_B, s_p_A_x_B])
row3 = widgets.HBox([slider_n2, s_p_A,       s_p_B])
row4 = widgets.HBox([slider_n3, s_p_A_giv_B, s_p_B_giv_A])
row5 = widgets.HBox([slider_n4, s_p_A_or_B,  s_p_neither])
row6 = widgets.HBox([slider_omega])

ui = widgets.VBox([
    widgets.HTML("<b>1. Räumliche Grenzen verschieben (Sample bleibt fix):</b>"),
    row1,
    widgets.HTML("<br><b>2. Wahrscheinlichkeiten ablesen & neue Punkte ziehen:</b>"),
    row2, row3, row4, row5, row6
])

In [4]:
sync_all_and_plot()
display(ui, plot_output)

Output()